# Ensambles y tuning — AP_2026Q2

Objetivo: combinar modelos con errores distintos y buscar hiperparámetros sin reutilizar el holdout para entrenar el meta-modelo.

Modelos base previstos: CatBoost profundo + SVD, CatBoost sin SVD, MLP tabular, MLP texto y un modelo lineal TF-IDF.

## Estrategias

1. **Hard voting:** cada modelo vota una clase.
2. **Soft voting:** se promedian probabilidades.
3. **Weighted soft voting:** los pesos se eligen en una validación interna, no directamente en el holdout.
4. **Stacking:** un meta-modelo aprende a combinar las probabilidades OOF.
5. **Secuencial:** el modelo principal decide; un segundo modelo solo corrige casos con probabilidad cercana a 0,5.

La combinación solo es útil si los errores no son idénticos. Antes de combinar se calcula la correlación entre vectores de error.

In [1]:
import numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold


In [2]:
# Las matrices deben contener probabilidades OOF y de holdout/test.
# columns: p_catboost, p_catboost_svd, p_mlp_tabular, p_mlp_text, p_linear
def score(p, y, threshold=.5):
    pred=np.where(p>=threshold,'used','new')
    return {'accuracy':accuracy_score(y,pred), 'balanced_accuracy':balanced_accuracy_score(y,pred), 'macro_f1':f1_score(y,pred,average='macro')}

def error_correlation(P, y):
    E=(P>=.5)!=(np.asarray(y)=='used')[:,None]
    return pd.DataFrame(E).corr()


## Votación y pesos

Los pesos no deben optimizarse sobre el holdout final. Se buscan dentro del desarrollo con una grilla pequeña o una optimización restringida, y luego se congelan.

In [3]:
# Ejemplo de soft voting y búsqueda de pesos sobre validación interna
def weighted_vote(P, weights, threshold=.5):
    w=np.asarray(weights,dtype=float); w=w/w.sum(); return (P*w).sum(axis=1)

# candidatos = [(0.6,0.2,0.1,0.1), ...]
scores=[]
for w in candidatos:
    pv=weighted_vote(P_valid,w)
    scores.append((score(pv,y_valid)['accuracy'],w))
best_weights=max(scores)[1]


NameError: name 'candidatos' is not defined

## Stacking sin leakage

Cada probabilidad de entrenamiento del meta-modelo debe ser OOF: el modelo base nunca puede haber visto la etiqueta de esa fila. El holdout se reserva hasta el final. Un modelo logístico suele ser suficiente como meta-modelo y es más interpretable que otra red profunda.

In [ ]:
meta=LogisticRegression(C=.3, class_weight=None, max_iter=1000)
meta.fit(P_oof, y_dev)
p_stack=meta.predict_proba(P_holdout)[:,1]
score(p_stack,y_holdout)


## Combinación secuencial

La regla correcta es conservadora: el segundo modelo solo interviene cuando el primero está cerca del umbral y su probabilidad coincide con la dirección del segundo. Se prueban bandas de confianza, por ejemplo `abs(p_catboost - .5) < {0.02, .04, .06, .08}`.

In [ ]:
def sequential(p_main,p_aux,band=.05,alpha=.25,threshold=.5):
    mask=np.abs(p_main-.5)<band
    p=p_main.copy(); p[mask]=(1-alpha)*p_main[mask]+alpha*p_aux[mask]
    return p

# evaluar bandas y alpha únicamente en una validación interna


## Tuning

Se comparan tres enfoques:

- `RandomizedSearchCV`: rápido y reproducible para rangos amplios.
- TPE/Optuna: concentra evaluaciones donde observa mejores resultados; requiere una validación interna fija.
- búsqueda guiada por errores: prioriza `depth`, `l2_leaf_reg`, `learning_rate`, `random_strength`, `bagging_temperature`, `border_count` y número de componentes SVD.

No se selecciona el mejor ensayo mirando repetidamente el holdout final.

In [ ]:
param_dist={
 'depth':[6,7,8,9], 'learning_rate':[.02,.03,.04,.05],
 'l2_leaf_reg':[3,6,10,20], 'random_strength':[0, .5, 1, 2],
 'bagging_temperature':[0, .5, 1, 2], 'border_count':[32,64,128]
}
# search=RandomizedSearchCV(estimator, param_dist, n_iter=20,
#     scoring='accuracy', cv=inner_cv, random_state=42, n_jobs=1)


## Tabla final y submission

Se conserva una tabla con modelo base, método de combinación, score de validación interna, score del holdout, umbral y pesos. Solo el ganador congelado genera la submission con `ID=1..30000`.

In [ ]:
def make_submission(pred, path):
    out=pd.DataFrame({'ID':np.arange(1,len(pred)+1),'condition':np.where(pred>=.5,'used','new')})
    assert len(out)==30000 and out.ID.is_unique
    out.to_csv(path,index=False)
    return out.head()
